In [1]:
!python helpers/generate_poisson_ar1.py

Simulated DataFrame:
              timestamp  y1  y2
0  2024-01-01T00:00:00Z   4   3
1  2024-01-01T01:00:00Z   6   6
2  2024-01-01T02:00:00Z   2   6
3  2024-01-01T03:00:00Z   1   3
4  2024-01-01T04:00:00Z   3   1
Simulated data saved to 'simulated_poisson_ar1.parquet'


In [2]:
# from raw data to features
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("MyApp") \
    .master("local[*]") \
    .getOrCreate()


In [3]:
# read from parquet file
df_sim = spark.read.parquet("simulated_poisson_ar1.parquet")
df_sim.show(5)

+--------------------+---+---+
|           timestamp| y1| y2|
+--------------------+---+---+
|2024-01-01T00:00:00Z|  4|  3|
|2024-01-01T01:00:00Z|  6|  6|
|2024-01-01T02:00:00Z|  2|  6|
|2024-01-01T03:00:00Z|  1|  3|
|2024-01-01T04:00:00Z|  3|  1|
+--------------------+---+---+
only showing top 5 rows



In [4]:
from pyspark.sql import functions as sf
from pyspark.sql.window import Window

# compute lagged values
df_sim = df_sim.withColumn("y1_lag1", sf.lag("y1", 1).over(Window.orderBy("timestamp")))
df_sim = df_sim.withColumn("y2_lag1", sf.lag("y2", 1).over(Window.orderBy("timestamp")))
df_sim.show(3)
df_sim.printSchema()

+--------------------+---+---+-------+-------+
|           timestamp| y1| y2|y1_lag1|y2_lag1|
+--------------------+---+---+-------+-------+
|2024-01-01T00:00:00Z|  4|  3|   NULL|   NULL|
|2024-01-01T01:00:00Z|  6|  6|      4|      3|
|2024-01-01T02:00:00Z|  2|  6|      6|      6|
+--------------------+---+---+-------+-------+
only showing top 3 rows

root
 |-- timestamp: string (nullable = true)
 |-- y1: long (nullable = true)
 |-- y2: long (nullable = true)
 |-- y1_lag1: long (nullable = true)
 |-- y2_lag1: long (nullable = true)



In [5]:
# filter NULL values and save to features.parquet
df_sim_clean = df_sim.filter(sf.col("y1_lag1").isNotNull() & sf.col("y2_lag1").isNotNull())
df_sim_clean.show(3)
df_sim_clean.printSchema()

+--------------------+---+---+-------+-------+
|           timestamp| y1| y2|y1_lag1|y2_lag1|
+--------------------+---+---+-------+-------+
|2024-01-01T01:00:00Z|  6|  6|      4|      3|
|2024-01-01T02:00:00Z|  2|  6|      6|      6|
|2024-01-01T03:00:00Z|  1|  3|      2|      6|
+--------------------+---+---+-------+-------+
only showing top 3 rows

root
 |-- timestamp: string (nullable = true)
 |-- y1: long (nullable = true)
 |-- y2: long (nullable = true)
 |-- y1_lag1: long (nullable = true)
 |-- y2_lag1: long (nullable = true)



In [6]:
# divide in features and labels
df_features = df_sim_clean.select("timestamp", "y1_lag1", "y2_lag1")
df_features.show(3)
df_features.printSchema()

df_label = df_sim_clean.select("timestamp", "y1", "y2")
df_label.show(3)
df_label.printSchema()

+--------------------+-------+-------+
|           timestamp|y1_lag1|y2_lag1|
+--------------------+-------+-------+
|2024-01-01T01:00:00Z|      4|      3|
|2024-01-01T02:00:00Z|      6|      6|
|2024-01-01T03:00:00Z|      2|      6|
+--------------------+-------+-------+
only showing top 3 rows

root
 |-- timestamp: string (nullable = true)
 |-- y1_lag1: long (nullable = true)
 |-- y2_lag1: long (nullable = true)

+--------------------+---+---+
|           timestamp| y1| y2|
+--------------------+---+---+
|2024-01-01T01:00:00Z|  6|  6|
|2024-01-01T02:00:00Z|  2|  6|
|2024-01-01T03:00:00Z|  1|  3|
+--------------------+---+---+
only showing top 3 rows

root
 |-- timestamp: string (nullable = true)
 |-- y1: long (nullable = true)
 |-- y2: long (nullable = true)



In [7]:
from pyspark.sql.functions import regexp_replace, col

# melt features dataframe
df_features_store = df_features.melt(ids=['timestamp'], values=['y1_lag1', 'y2_lag1'],
                                      variableColumnName='entity_id', 
                                      valueColumnName='lag_count')

# replace 'y1_lag', 'y2_lag' with 'y1', 'y2' respectively
df_features_store = df_features_store.withColumn('entity_id', regexp_replace('entity_id', 'y1_lag1', 'y1')) \
                                        .withColumn('entity_id', regexp_replace('entity_id', 'y2_lag1', 'y2'))



# rename column
df_features_store = df_features_store.withColumnRenamed("timestamp", "event_timestamp")

# cast to timestamp
df_features_store = df_features_store.withColumn(
    "event_timestamp", col("event_timestamp").cast("timestamp")
)

df_features_store.show(3)
df_features_store.printSchema()

+-------------------+---------+---------+
|    event_timestamp|entity_id|lag_count|
+-------------------+---------+---------+
|2024-01-01 01:00:00|       y1|        4|
|2024-01-01 01:00:00|       y2|        3|
|2024-01-01 02:00:00|       y1|        6|
+-------------------+---------+---------+
only showing top 3 rows

root
 |-- event_timestamp: timestamp (nullable = true)
 |-- entity_id: string (nullable = false)
 |-- lag_count: long (nullable = true)



In [8]:
# melt label dataframe
df_label_store = df_label.melt(ids=['timestamp'], values=['y1', 'y2'],
                                      variableColumnName='entity_id', 
                                      valueColumnName='target')
# same as for features
df_label_store = df_label_store.withColumnRenamed("timestamp", "event_timestamp")
df_label_store = df_label_store.withColumn(
    "event_timestamp", col("event_timestamp").cast("timestamp")
)
df_label_store.show(3)
df_label_store.printSchema()

+-------------------+---------+------+
|    event_timestamp|entity_id|target|
+-------------------+---------+------+
|2024-01-01 01:00:00|       y1|     6|
|2024-01-01 01:00:00|       y2|     6|
|2024-01-01 02:00:00|       y1|     2|
+-------------------+---------+------+
only showing top 3 rows

root
 |-- event_timestamp: timestamp (nullable = true)
 |-- entity_id: string (nullable = false)
 |-- target: long (nullable = true)



In [9]:
# save both features and label as parquet
df_features_store.select("event_timestamp", "entity_id", "lag_count").write.mode("overwrite").parquet("features.parquet")
df_label_store.select("event_timestamp", "entity_id", "target").write.mode("overwrite").parquet("label.parquet")

In [24]:
#import sys
#!pip install feast
#!{sys.executable} -m pip install \
#    "numpy>=2.0,<3" \
#    "pandas>=2.0,<3" \
#    "numba>=0.60" \
#    "scipy>=1.13" \
#    "scikit-learn>=1.4" \
#    "matplotlib>=3.9" \
#    --force-reinstall

#!{sys.executable} -m pip install --upgrade numexpr bottleneck

In [26]:
import pandas as pd
import numpy as np
print(pd.__version__, np.__version__)

2.3.3 2.4.4


In [68]:
from feast import FeatureStore

store = FeatureStore(repo_path="feature_repo/feature_repo")

entity_df = pd.read_parquet("feature_repo/feature_repo/data/label.parquet")
entity_df = entity_df[entity_df["entity_id"] == "y1"]
entity_df_train = entity_df.iloc[:-50]
entity_df_test = entity_df.iloc[-50:][["event_timestamp", "entity_id"]]

,event_timestamp,entity_id,target
0,2024-01-01 01:00:00,y1,6
2,2024-01-01 02:00:00,y1,2
4,2024-01-01 03:00:00,y1,1


In [73]:
training_df = store.get_historical_features(
    entity_df=entity_df_train,
    features=[
        "lag_features:lag_count",
    ],
).to_df()

print("length", len(training_df))
print(training_df.tail())

length 249
              event_timestamp entity_id  target  lag_count
244 2024-01-11 05:00:00+00:00        y1       3          0
245 2024-01-11 06:00:00+00:00        y1       5          3
246 2024-01-11 07:00:00+00:00        y1       0          5
247 2024-01-11 08:00:00+00:00        y1       3          0
248 2024-01-11 09:00:00+00:00        y1       1          3


In [72]:
testing_df = store.get_historical_features(
    entity_df=entity_df_test,
    features=[
        "lag_features:lag_count",
    ],
).to_df()

print("length", len(testing_df))
print(testing_df.head())

length 50
            event_timestamp entity_id  lag_count
0 2024-01-11 10:00:00+00:00        y1          1
1 2024-01-11 11:00:00+00:00        y1          1
2 2024-01-11 12:00:00+00:00        y1          0
3 2024-01-11 13:00:00+00:00        y1          2
4 2024-01-11 14:00:00+00:00        y1          1
